# HTML2Video — My Voice pack (Colab, free)

Is notebook se aapki **apni awaaz** mein voice-over clips banti hain, jo HTML2Video app mein **My Voice → Voice pack import** se judti hain.

**Zaroori baatein**
- Sirf **apni** awaaz (ya jiski permission ho) use karein.
- Colab free GPU kabhi-kabhi nahi milta. Runtime → Change runtime type → **T4 GPU** chuno.
- Mobile browser mein: Chrome menu → **Desktop site** on karke notebook kholo.
- Ye notebook maine yahan chala kar test nahi kiya (mere paas GPU/Colab nahi hai). Install ya version error aaye to neeche "Agar error aaye" dekho.

**Steps**
1. HTML2Video app: HTML load karo → engine **My Voice** → **Script export** (Download/HTML2Video/`my_voice_script.json`).
2. Apni awaaz ka 1-3 minute ka saaf sample record karo (shor-free, ek hi mic, normal speed).
3. Neeche ke cells upar se niche chalao (Shift+Enter). Jab `files.upload()` aaye to **script.json** aur **voice sample** dono upload karo.
4. End mein `my_voice_pack.zip` download hogi. App mein **Voice pack import** se us zip ko chuno.

**MODE**
- `convert` (recommended): Edge TTS se Hinglish clips banti hain, phir **OpenVoice v2** unhe aapki tone mein badalta hai. Pronunciation stable rehti hai.
- `xtts`: XTTS-v2 seedha aapki awaaz clone karta hai. **License non-commercial hai** (monetized YouTube ke liye khud check karo). Hinglish (Roman Hindi) kabhi galat padh sakta hai.

In [ ]:
# ---- SETTINGS ----
MODE = "convert"                 # "convert" ya "xtts"
BASE_VOICE = "en-IN-PrabhatNeural"   # Hinglish (Roman Hindi) ke liye Indian-English voice. Female: "en-IN-NeerjaNeural"
                                     # Pure Hindi text ho to: "hi-IN-MadhurNeural" / "hi-IN-SwaraNeural"
BASE_RATE = "+0%"                # speed: "-10%" dheema, "+10%" tez

In [ ]:
!pip -q install edge-tts pydub soundfile huggingface_hub
!apt-get -qq install -y ffmpeg > /dev/null
print("install ok")

In [ ]:
# ---- UPLOAD: my_voice_script.json + apni awaaz ka sample ----
from google.colab import files
import os, json
up = files.upload()
script_path = next((n for n in up if n.lower().endswith(".json")), None)
sample_path = next((n for n in up if n.lower().endswith((".wav",".mp3",".m4a",".aac",".ogg",".flac"))), None)
assert script_path and sample_path, "script.json aur apni awaaz ka sample, dono upload karo"
script = json.load(open(script_path, encoding="utf-8"))
print(len(script), "lines mili. Pehli line:", script[0]["text"][:80])
!ffmpeg -y -loglevel error -i "{sample_path}" -ac 1 -ar 22050 ref.wav
print("reference ready: ref.wav")

## A) MODE = convert  (Edge TTS base  →  OpenVoice v2 se aapki tone)

In [ ]:
# ---- 1) base clips (Edge TTS) ----
import asyncio, edge_tts, subprocess, os
os.makedirs("base", exist_ok=True)

async def gen(item):
    n = int(item["n"]); mp3 = f"base/{n:03d}.mp3"; wav = f"base/{n:03d}.wav"
    if os.path.exists(wav): return
    await edge_tts.Communicate(item["text"], BASE_VOICE, rate=BASE_RATE).save(mp3)
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",mp3,"-ac","1","-ar","24000",wav], check=True)

async def run_all():
    sem = asyncio.Semaphore(4)
    async def w(it):
        async with sem: await gen(it)
    await asyncio.gather(*(w(it) for it in script))

if MODE == "convert":
    await run_all()
    print("base clips ready:", len(os.listdir("base")) // 2)

In [ ]:
# ---- 2) OpenVoice v2 setup ----
if MODE == "convert":
    !git clone -q https://github.com/myshell-ai/OpenVoice.git
    !pip -q install -e OpenVoice --no-deps
    !pip -q install librosa faster-whisper whisper-timestamped wavmark eng_to_ipa inflect unidecode pypinyin cn2an jieba langid
    from huggingface_hub import snapshot_download
    snapshot_download(repo_id="myshell-ai/OpenVoiceV2", local_dir="checkpoints_v2")
    print("OpenVoice ready")

In [ ]:
# ---- 3) convert: base voice -> aapki tone ----
SRC_DIR = None
if MODE == "convert":
    import sys, glob, torch
    sys.path.insert(0, "OpenVoice")
    from openvoice import se_extractor
    from openvoice.api import ToneColorConverter
    from pydub import AudioSegment

    device = "cuda" if torch.cuda.is_available() else "cpu"
    conv = ToneColorConverter("checkpoints_v2/converter/config.json", device=device)
    conv.load_ckpt("checkpoints_v2/converter/checkpoint.pth")

    target_se, _ = se_extractor.get_se("ref.wav", conv, vad=False)

    # base voice ka tone nikalne ke liye kuch base clips jodte hain (~20+ sec)
    seg = AudioSegment.empty()
    for f in sorted(glob.glob("base/*.wav")):
        seg += AudioSegment.from_wav(f)
        if len(seg) > 30000: break
    seg.export("base_ref.wav", format="wav")
    source_se, _ = se_extractor.get_se("base_ref.wav", conv, vad=False)

    os.makedirs("conv", exist_ok=True)
    for it in script:
        n = int(it["n"])
        conv.convert(audio_src_path=f"base/{n:03d}.wav", src_se=source_se, tgt_se=target_se,
                     output_path=f"conv/{n:03d}.wav", message="@MyVoice")
        if n % 10 == 0: print("converted", n, "/", len(script))
    SRC_DIR = "conv"
    print("convert done")

## B) MODE = xtts  (seedha clone — license non-commercial)

In [ ]:
if MODE == "xtts":
    !pip -q install coqui-tts
    import torch, os
    from TTS.api import TTS
    device = "cuda" if torch.cuda.is_available() else "cpu"
    tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to(device)
    os.makedirs("xtts", exist_ok=True)
    for it in script:
        n = int(it["n"])
        tts.tts_to_file(text=it["text"], speaker_wav="ref.wav", language="hi", file_path=f"xtts/{n:03d}.wav")
        if n % 10 == 0: print("xtts", n, "/", len(script))
    SRC_DIR = "xtts"
    print("xtts done")

## Pack banao aur download karo

In [ ]:
import shutil, subprocess, os
assert SRC_DIR, "Pehle upar ka MODE wala section chalao"
os.makedirs("pack", exist_ok=True)
for it in script:
    n = int(it["n"])
    subprocess.run(["ffmpeg","-y","-loglevel","error","-i",f"{SRC_DIR}/{n:03d}.wav",
                    "-ac","1","-ar","48000","-sample_fmt","s16",f"pack/{n:03d}.wav"], check=True)
shutil.make_archive("my_voice_pack", "zip", "pack")
print("my_voice_pack.zip ban gayi:", len(os.listdir("pack")), "clips")
from google.colab import files
files.download("my_voice_pack.zip")

## Agar error aaye
- **pip / version conflict:** Runtime → Restart session, phir sab cells dobara chalao. OpenVoice ki install repo update hone par badal sakti hai.
- **OpenVoice hi na chale:** `MODE = "xtts"` try karo (par license dekho), ya app mein **Record lines** se khud record karo.
- **Awaaz thodi robotic/galat pronunciation:** `BASE_VOICE` badlo (`en-IN-NeerjaNeural`, `hi-IN-MadhurNeural`), ya sample zyada saaf/lamba (2-3 min) do.
- **App mein import ke baad lines kam dikhein:** zip ki file names `001.wav`, `002.wav` ... hone chahiye (script ke `n` ke hisaab se). Script wahi use karo jo app se export ki thi.